# Merge Binance Daily + Onchain (4 coins)

This notebook merges Binance 1d data with onchain features for:
- btc
- eth
- xrp
- doge

Only required onchain columns are merged:
- `AdrActCnt`
- `CapMVRVCur`

Then it removes **leading NaNs** using these required columns.

BNB is intentionally excluded.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError(f'Cannot locate project data directory from cwd={Path.cwd()}')

print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
ASSETS = ['btc', 'eth', 'xrp', 'doge']
REQUIRED_OC_COLS = ['AdrActCnt', 'CapMVRVCur']

BINANCE_TEMPLATE = PROJECT_ROOT / 'data' / '{asset}-1d.csv'
ONCHAIN_TEMPLATE = PROJECT_ROOT / 'data' / 'onchain' / '{asset}.csv'

OUT_RAW_DIR = PROJECT_ROOT / 'data' / 'merged_onchain_1d_raw'
OUT_CLEAN_DIR = PROJECT_ROOT / 'data' / 'merged_onchain_1d'
OUT_RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_CLEAN_DIR.mkdir(parents=True, exist_ok=True)

check_rows = []
for asset in ASSETS:
    b = Path(str(BINANCE_TEMPLATE).format(asset=asset))
    o = Path(str(ONCHAIN_TEMPLATE).format(asset=asset))
    check_rows.append(
        {
            'asset': asset,
            'binance_exists': b.exists(),
            'onchain_exists': o.exists(),
            'binance': str(b),
            'onchain': str(o),
        }
    )

check_df = pd.DataFrame(check_rows)
display(check_df)

if not check_df['binance_exists'].all() or not check_df['onchain_exists'].all():
    missing = check_df[(~check_df['binance_exists']) | (~check_df['onchain_exists'])]
    raise FileNotFoundError('Missing source files\n' + missing.to_string(index=False))


In [ ]:
def _to_day_utc(series) -> pd.Series:
    dt = pd.to_datetime(series, utc=True, errors='coerce')
    dt = dt.dt.tz_convert(None).dt.normalize()
    return dt


def load_binance_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if 'datetime_utc' not in df.columns:
        raise ValueError(f"Missing 'datetime_utc' in {path}")
    df = df.copy()
    df['_date'] = _to_day_utc(df['datetime_utc'])
    df = df.dropna(subset=['_date']).sort_values('_date').drop_duplicates('_date', keep='last')
    return df


def load_onchain_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if 'time' not in df.columns:
        raise ValueError(f"Missing 'time' in {path}")
    df = df.copy()
    df['_date'] = _to_day_utc(df['time'])
    df = df.dropna(subset=['_date']).sort_values('_date').drop_duplicates('_date', keep='last')
    return df


def merge_and_trim_leading_nans(asset: str, required_cols: list[str]):
    b_path = Path(str(BINANCE_TEMPLATE).format(asset=asset))
    o_path = Path(str(ONCHAIN_TEMPLATE).format(asset=asset))

    bdf = load_binance_daily(b_path)
    odf = load_onchain_daily(o_path)

    missing_required_in_onchain = [c for c in required_cols if c not in odf.columns]
    if missing_required_in_onchain:
        raise ValueError(f"{asset}: missing required onchain columns in source file: {missing_required_in_onchain}")

    merged = bdf.merge(odf[['_date'] + required_cols], on='_date', how='left')

    if merged['_date'].isna().any():
        raise ValueError(f'{asset}: unexpected NaT in merged _date')

    merged['datetime_utc'] = merged['_date'].dt.strftime('%Y-%m-%dT00:00:00Z')

    valid_mask = merged[required_cols].notna().all(axis=1)
    if valid_mask.any():
        first_valid_pos = int(np.argmax(valid_mask.to_numpy()))
        cleaned = merged.iloc[first_valid_pos:].copy()
    else:
        first_valid_pos = None
        cleaned = merged.iloc[0:0].copy()

    out_cols = ['datetime_utc'] + [c for c in merged.columns if c not in {'datetime_utc', '_date'}]
    merged = merged[out_cols].copy()
    cleaned = cleaned[out_cols].copy()

    raw_out = OUT_RAW_DIR / f'{asset}-1d-merged-raw.csv'
    clean_out = OUT_CLEAN_DIR / f'{asset}-1d-merged.csv'
    merged.to_csv(raw_out, index=False)
    cleaned.to_csv(clean_out, index=False)

    info = {
        'asset': asset,
        'raw_rows': len(merged),
        'clean_rows': len(cleaned),
        'dropped_leading_rows': len(merged) - len(cleaned),
        'first_raw_date': merged['datetime_utc'].iloc[0] if len(merged) else None,
        'first_clean_date': cleaned['datetime_utc'].iloc[0] if len(cleaned) else None,
        'last_clean_date': cleaned['datetime_utc'].iloc[-1] if len(cleaned) else None,
        'required_nan_in_clean': int(cleaned[required_cols].isna().any(axis=1).sum()) if len(cleaned) else None,
        'raw_out': str(raw_out),
        'clean_out': str(clean_out),
        'first_valid_pos': first_valid_pos,
    }
    return merged, cleaned, info


In [ ]:
raw_by_asset = {}
clean_by_asset = {}
summary_rows = []

for asset in ASSETS:
    merged_df, clean_df, info = merge_and_trim_leading_nans(asset, REQUIRED_OC_COLS)
    raw_by_asset[asset] = merged_df
    clean_by_asset[asset] = clean_df
    summary_rows.append(info)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


In [ ]:
# Quick sanity preview per asset
for asset in ASSETS:
    print('\n' + '=' * 80)
    print('ASSET:', asset)

    preview_cols = ['datetime_utc'] + REQUIRED_OC_COLS
    preview_cols = [c for c in preview_cols if c in raw_by_asset[asset].columns]

    print('RAW head:')
    display(raw_by_asset[asset][preview_cols].head(5))

    print('CLEAN head:')
    display(clean_by_asset[asset][preview_cols].head(5))
